# Chapter 3: File Formats

*From: Database Internals*

---

## TL;DR

- On-disk structures have no `malloc`/`free` — we read and write byte offsets, so layout is everything. Pages are the unit of I/O (typically 4–16 KB) and most engines fix the page size for the life of the file.
- Primitive types have fixed sizes and a chosen endianness; variable-size data (strings, arrays) is written as a **length prefix + raw bytes** (Pascal-style); booleans and flags are **bit-packed** behind bitmasks.
- **Slotted pages** solve variable-size record storage by growing a cell-pointer array from one end and the cells themselves from the other, meeting in a free-space gap — insertion is append-only, logical order lives in the pointer array.
- Two cell flavors cover B-Trees: **key cells** (separator key + child page ID) in internal nodes and **key-value cells** (key + value bytes) in leaves — both are `header | key | value`.
- Deletion is lazy: freed cells go on an **availability list** (SQLite's freeblocks) and are reused via first-fit or best-fit; when no single block is big enough the page is defragmented, and only then is an overflow page allocated.
- Cross-cutting concerns live in the header: a **version identifier** (filename prefix, sidecar file, or in-header magic number) and a **page-level checksum** (CRC, not plain XOR parity) for corruption detection.

---

## The Problem

- **No memory allocator.** Unlike main memory, disks give you `read` and `write` over byte offsets — no `malloc`, no GC. Every byte of layout, alignment, and free space is your responsibility.
- **I/O is page-sized and random is slow.** You pay to bring a whole page into RAM whether you need one record or hundreds. Layout decisions are judged by how much useful work one page read delivers.
- **Files outlive code.** Readers must parse files written by older binary versions. The format has to be self-describing *before* the reader knows how to interpret the rest of it.

---

## Binary Encoding Primitives

Before we can lay out pages, we have to serialize single values into bytes. Three ideas do most of the work.

### Endianness

Multibyte numeric types are written MSB-first (**big-endian**) or LSB-first (**little-endian**). Both ends of the wire must agree. A 32-bit integer `0xAABBCCDD` ends up as:

```
big-endian:    AA BB CC DD     (MSB at lowest address)
little-endian: DD CC BB AA     (LSB at lowest address)
```

Fixed sizes for integer primitives: `byte = 1`, `short = 2`, `int = 4`, `long = 8`. Floats follow IEEE 754 (`float = 4`, `double = 8`: sign + exponent + fraction).

### Pascal strings (length-prefixed)

Strings and variable-size arrays are written as a length prefix followed by that many bytes:

```
String {
    size  uint_16
    data  byte[size]
}
```

- **Pros vs C-style null-terminated strings:** O(1) length lookup, cheap slicing, no terminator byte to find or escape.
- **Con:** a 2-byte length caps string size at 65,535 — some formats use 4-byte lengths or variable-length (varint) prefixes.

### Bit-packed flags

A boolean needs one bit, not one byte. Pack up to 8 booleans into a byte and address them with masks. Flag bits are always powers of two so exactly one bit is set per mask:

```
IS_LEAF_MASK         = 0x01  // bit 0
VARIABLE_SIZE_VALUES = 0x02  // bit 1
HAS_OVERFLOW_PAGES   = 0x04  // bit 2
```

Set with `flags |= MASK`, clear with `flags &= ~MASK`, test with `(flags & MASK) != 0`.

A small runnable example — endianness conversion and flag manipulation:

In [ ]:
import struct

value = 0xAABBCCDD

# Same 32-bit integer in both byte orders
big = struct.pack(">I", value)      # network / big-endian
little = struct.pack("<I", value)   # x86-native / little-endian

print("big-endian bytes   :", big.hex(" "))
print("little-endian bytes:", little.hex(" "))

# Round-trip: unpack little-endian back to the integer
back = struct.unpack("<I", little)[0]
print(f"round-trip: 0x{back:08X}  (match={back == value})")

# --- Bit-packed page flags ---
IS_LEAF              = 0x01
VARIABLE_SIZE_VALUES = 0x02
HAS_OVERFLOW_PAGES   = 0x04

flags = 0
flags |= IS_LEAF | VARIABLE_SIZE_VALUES   # set two flags

def describe(f):
    return {
        "IS_LEAF":              bool(f & IS_LEAF),
        "VARIABLE_SIZE_VALUES": bool(f & VARIABLE_SIZE_VALUES),
        "HAS_OVERFLOW_PAGES":   bool(f & HAS_OVERFLOW_PAGES),
    }

print("flags byte :", f"0x{flags:02X}  bits={flags:08b}")
print("decoded    :", describe(flags))

flags &= ~IS_LEAF                         # clear the leaf bit
print("after clear:", f"0x{flags:02X}  bits={flags:08b}")

---

## Storage Math: Cells per Page

Every layout decision — key size, page size, whether values live in leaves — ultimately shows up as **fan-out** (cells per internal page) and **records per leaf**. Fan-out sets tree height; tree height sets I/O per point lookup.

The Python cell below computes, for a few realistic configurations:

- **Fan-out** of an internal page holding variable-size *key cells* (`key_size | page_id | key_bytes`).
- **Records per leaf** for *key-value cells* (`flags | key_size | value_size | key | value`).
- **Slotted-page overhead %** — how much of the page is header + pointer array vs. actual payload.
- **Tree height** for a target row count at the computed fan-out.

In [ ]:
import math

PAGE_HEADER_BYTES = 24        # magic, flags, checksum, free-space ptr, num-cells, etc.
CELL_POINTER_BYTES = 2        # one 16-bit offset per cell in the slot directory

# --- Key cell header (internal node): key_size(2) + page_id(4) ---
KEY_CELL_FIXED = 2 + 4

# --- Key-value cell header (leaf node): flags(1) + key_size(2) + value_size(2) ---
KV_CELL_FIXED = 1 + 2 + 2

def cells_per_page(page_size, cell_body_bytes):
    """How many cells fit in a slotted page?

    Each cell consumes (cell header + body) bytes in the cell region
    AND one CELL_POINTER_BYTES entry in the slot directory.
    """
    usable = page_size - PAGE_HEADER_BYTES
    per_cell = cell_body_bytes + CELL_POINTER_BYTES
    return usable // per_cell

def slotted_overhead_pct(page_size, n_cells, cell_body_bytes):
    """Fraction of the page NOT occupied by cell payload."""
    payload = n_cells * cell_body_bytes
    overhead = page_size - payload
    return 100.0 * overhead / page_size

def tree_height(rows, fanout, leaf_records):
    """Levels needed to index `rows` records with given internal fan-out
    and leaf capacity. Root counts as level 1."""
    leaves = math.ceil(rows / leaf_records)
    h = 1
    nodes = leaves
    while nodes > 1:
        nodes = math.ceil(nodes / fanout)
        h += 1
    return h

# --- Scenario: 8-byte keys, 8-byte page IDs on internal nodes;
#               8-byte keys, 64-byte values on leaves ---
KEY_BYTES   = 8
VALUE_BYTES = 64

internal_cell_body = KEY_CELL_FIXED + KEY_BYTES           # header + key bytes
leaf_cell_body     = KV_CELL_FIXED  + KEY_BYTES + VALUE_BYTES

print(f"{'page':>6} | {'fanout':>7} | {'leaf recs':>9} | {'leaf overhead':>14}")
print("-" * 48)
for page_size in (4096, 8192, 16384):
    fanout     = cells_per_page(page_size, internal_cell_body)
    leaf_recs  = cells_per_page(page_size, leaf_cell_body)
    overhead   = slotted_overhead_pct(page_size, leaf_recs, leaf_cell_body)
    print(f"{page_size:>6} | {fanout:>7} | {leaf_recs:>9} | {overhead:>13.1f}%")

# --- Tree height for 1 billion rows on an 8 KB page ---
page_size = 8192
fanout    = cells_per_page(page_size, internal_cell_body)
leaf_recs = cells_per_page(page_size, leaf_cell_body)
rows      = 1_000_000_000

h = tree_height(rows, fanout, leaf_recs)
print(f"\n8 KB page, fanout={fanout}, leaf_recs={leaf_recs}")
print(f"1B rows -> tree height = {h}  (= I/Os per point lookup, cold cache)")

Key observations:

- Fan-out is overwhelmingly dominated by **key size**, not page size — doubling the page doubles fan-out but only shaves `log(2)` off the height.
- Slotted-page overhead is roughly `(page_header + 2 * n_cells) / page_size` — a few percent for 8 KB pages with small cells, more painful for very small cells.
- At typical B-Tree fan-outs (hundreds), **three or four levels** index a billion rows. That's the whole point.

---

## Slotted Page Layout

A slotted page is the standard answer for "how do I pack variable-size records into a fixed-size page and still address them cheaply?"

Three requirements drive the design:

1. Store variable-size records with minimal overhead.
2. Reclaim space freed by deletions without breaking external references.
3. Reference records by a stable ID, not an absolute offset.

The trick: put a fixed-size **header** at the top, grow a **pointer/slot array** rightward from just after the header, grow the **cells** leftward from the end of the page, and let free space sit between them.

```mermaid
graph TB
    subgraph page["Slotted Page"]
        H["Header (fixed size)<br/>magic, flags, checksum,<br/>num_cells, free_ptr"]
        P["Slot / Pointer Array -->"]
        F["&lt;-- Free space --&gt;"]
        C["&lt;-- Cells (variable size)"]
    end
    H --> P
    P --> F
    F --> C
```

Conceptually, the byte ranges look like:

```
offset 0                                                  page_size
+----------+----+----+----+-----+----------+---------+---------+
| header   | p0 | p1 | p2 | ... |  FREE    |  cell2  |  cell0  |
+----------+----+----+----+-----+----------+---------+---------+
           ^^^^^^^^^^^^^^^^                          ^^^^^^^^
         pointers grow right                     cells grow left
```

- **Insertion** appends a new cell at the high-water mark of the cells region and writes a new slot entry. Cells never move on insert.
- **Logical order** (e.g., key order) is maintained by keeping the **slot array sorted by key** — slot entries shift on insert, actual cells don't. This lets us binary-search.
- **Deletion** nulls (or removes) the slot and marks the cell's bytes as free — the cell stays where it is.
- **External pointers** reference cells by `(page_id, slot_index)`, not by byte offset, so compaction and rearrangement inside the page are invisible to outsiders.

Example insertion order vs. logical order: inserting `Tom`, `Leslie`, `Ron` leaves cells physically in insertion order but the slot array is `[Leslie, Ron, Tom]` so a binary search on keys still works.

---

## Cell Layout Deep Dive

B-Tree cells come in exactly two flavors, and a page is homogeneous (all cells on a page are the same flavor). That homogeneity lets the page header describe cell shape once, instead of every cell duplicating its type.

### Key cell (internal node)

Used on internal / non-leaf nodes. Stores a **separator key** and a **child page ID** pointing to the subtree for keys ≥ this separator.

```
byte 0          2               6              6 + key_size
+---------------+---------------+-----------------+
| [u16]         | [u32]         | [bytes]         |
| key_size      | page_id       | key bytes       |
+---------------+---------------+-----------------+
  <-- fixed-size header -->      <-- variable -->
```

- `key_size` is only needed when keys are variable-size; fixed-size keys drop it.
- `page_id` is an *ID*, not a byte offset — the buffer manager translates it to a file offset via a lookup table, so the ID stays compact (often 32 bits even for multi-TB files).

### Key-value cell (leaf node)

Used on leaf nodes. Stores the **key** and the **value bytes** (the actual record or a pointer into an overflow page if the record is too large).

```
byte 0      1               3                5
+-----------+---------------+---------------+
| [u8]      | [u16]         | [u16]         |
| flags     | key_size      | value_size    |
+-----------+---------------+---------------+
byte 5                     5 + key_size       5 + key_size + value_size
+---------------------------+---------------------+
| [bytes]                   | [bytes]             |
| key bytes                 | value bytes         |
+---------------------------+---------------------+
```

- `flags` carries per-cell bits (tombstone, has-overflow, etc.).
- Putting fixed-size fields first means locating the variable parts is just arithmetic: `key @ header_end`, `value @ header_end + key_size`.

### Key cell vs. key-value cell

| Field        | Key cell (internal)       | Key-value cell (leaf)       |
|--------------|---------------------------|-----------------------------|
| Fixed header | `key_size`, `page_id`     | `flags`, `key_size`, `value_size` |
| Variable     | `key`                     | `key`, `value`              |
| Reference    | Page ID -> another page   | Cell offset -> this page's cell region |
| Used in      | Non-leaf nodes            | Leaf nodes                  |
| Purpose      | Navigate the tree         | Hold the actual record      |

---

## Free-Space Management

Deleting a cell doesn't rewrite the page. Instead, the cell's `(offset, size)` goes onto an **availability list** (SQLite calls the entries *freeblocks*; the page header stores a pointer to the first freeblock and the total free byte count). On insert, we try to reuse a freeblock before bumping the free-space pointer.

Two classic strategies pick a freeblock:

- **First-fit** — scan the list, take the first block that's big enough. Fast, but the remainder is often too small to be useful, leading to fragmentation.
- **Best-fit** — scan the whole list, take the block whose leftover is smallest. Slower, tighter packing, less "dust" left behind.

If no single freeblock fits the new cell but the *total* free space would, we defragment: rewrite live cells contiguously and reset the availability list. If even that isn't enough, we spill to an **overflow page**.

The cell below simulates both strategies on a shared availability list and reports waste + fragmentation at the end:

In [ ]:
from copy import deepcopy

def first_fit(avail, size):
    """Return index of first freeblock >= size, or -1."""
    for i, block in enumerate(avail):
        if block >= size:
            return i
    return -1

def best_fit(avail, size):
    """Return index of smallest freeblock >= size, or -1."""
    best_i, best_sz = -1, None
    for i, block in enumerate(avail):
        if block >= size and (best_sz is None or block < best_sz):
            best_i, best_sz = i, block
    return best_i

def simulate(strategy_fn, freeblocks, inserts):
    """Run a placement strategy over a sequence of insert sizes.
    Returns (freeblocks_after, placed, failed)."""
    blocks = list(freeblocks)
    placed, failed = 0, 0
    for size in inserts:
        idx = strategy_fn(blocks, size)
        if idx < 0:
            failed += 1
            continue
        # Consume: replace the block with (block - size); drop if 0
        remainder = blocks[idx] - size
        if remainder == 0:
            blocks.pop(idx)
        else:
            blocks[idx] = remainder
        placed += 1
    return blocks, placed, failed

# A fragmented 8 KB page: initial availability list (freeblock sizes in bytes)
initial = [120, 48, 300, 72, 180, 24, 96, 500, 60]
# Insert requests to service (in order)
requests = [80, 50, 150, 40, 220, 60, 110]

for name, fn in [("first-fit", first_fit), ("best-fit", best_fit)]:
    after, placed, failed = simulate(fn, initial, requests)
    total_free_after = sum(after)
    dust = sum(b for b in after if b < 32)  # blocks too small to be useful
    n_blocks = len(after)
    print(f"{name:>9}: placed={placed}/{placed+failed}  "
          f"free_after={total_free_after:4d}B  "
          f"dust(<32B)={dust:3d}B  "
          f"blocks={n_blocks}  "
          f"sizes={after}")

Expected pattern: first-fit is quicker to decide but tends to leave smaller, more numerous leftover blocks ("dust"); best-fit minimizes immediate waste but a pathological best-fit run can still end up with many tiny unusable blocks over time. Real engines (SQLite, PostgreSQL) use variants of first-fit for speed and rely on page-level defragmentation to clean up.

---

## Versioning

File formats evolve, and readers must identify the version *before* they know how to parse the rest. Three common places to stash the version indicator:

| Technique             | Where the version lives            | Example            | Pros                                      | Cons                                          |
|-----------------------|------------------------------------|--------------------|-------------------------------------------|-----------------------------------------------|
| Filename prefix       | In the file's name                 | Cassandra SSTables (`na-1-big-Data.db`, older `ma-*`) | Readable without opening the file; trivial to filter on disk | Couples format to filesystem layout; renaming = corruption risk |
| Sidecar file          | In a separate, tiny file           | PostgreSQL `PG_VERSION`  | Single source of truth per cluster; no per-file overhead | Extra file to keep in sync; not useful for per-file variation |
| In-header magic number| First bytes of the file header     | Many engines (SQLite, Parquet, etc.) | Self-describing; survives rename/move; version-stable header bootstraps the reader | Must keep the *header* format stable across versions forever |

**Magic numbers** (a fixed, known-constant byte sequence at a fixed offset) double as version tags: the reader verifies the magic, dispatches to the correct parser, and only then interprets the rest.

---

## Checksumming

Disks lie occasionally — bit flips, torn writes, controller bugs. Software lies occasionally too. Catching corruption *at read time* keeps it from poisoning downstream caches, replicas, or query results.

### Checksum vs. CRC vs. cryptographic hash

| Property                        | XOR / parity checksum | CRC32 / CRC64         | Cryptographic hash (SHA-256) |
|---------------------------------|-----------------------|-----------------------|------------------------------|
| Detects single-bit errors       | Yes                   | Yes                   | Yes                          |
| Detects multi-bit / burst errors| Mostly no             | Yes (by design)       | Yes                          |
| Detects adversarial tampering   | No                    | No                    | Yes                          |
| Cost                            | Very cheap            | Cheap (table-driven)  | Expensive                    |
| Use case                        | Back-of-envelope      | On-disk page integrity| Authenticated storage / signing |

For accidental corruption on disk, **CRC is the right default**: cheap, lookup-table based, proven to catch burst errors which dominate real-world disk and network failures. It is **not** a substitute for a cryptographic hash when the threat model is tampering.

### Where the checksum lives

Computing a checksum over the entire file on every read is wasteful. Instead, each page stores its own checksum **in the page header**:

- **Write path:** compute CRC over the page body, write `header.checksum = CRC`, then write the page.
- **Read path:** read the page, recompute CRC over the body, compare against `header.checksum`. Mismatch → raise a corruption error *before* the page reaches the buffer pool.

Per-page checksums also bound the blast radius: a single corrupt page is a single bad record region, not a dead file.

In [ ]:
import zlib

page_body = b"slotted page payload: key0|v0 key1|v1 key2|v2 ..." * 64   # ~3 KB

# Write path
checksum = zlib.crc32(page_body) & 0xFFFFFFFF    # unsigned 32-bit
print(f"CRC32 of page body: 0x{checksum:08X}  ({len(page_body)} bytes)")

# Simulate the page on disk: 4-byte CRC header + body
on_disk = checksum.to_bytes(4, "big") + page_body

# Read path — happy case
stored_crc = int.from_bytes(on_disk[:4], "big")
recomputed = zlib.crc32(on_disk[4:]) & 0xFFFFFFFF
print(f"read-back OK? {stored_crc == recomputed}")

# Read path — inject a one-byte corruption somewhere in the body
corrupted = bytearray(on_disk)
corrupted[500] ^= 0x01                            # flip one bit
stored_crc = int.from_bytes(corrupted[:4], "big")
recomputed = zlib.crc32(corrupted[4:]) & 0xFFFFFFFF
print(f"corrupted detected? {stored_crc != recomputed}  "
      f"(stored=0x{stored_crc:08X}, recomputed=0x{recomputed:08X})")

A single-bit flip changes the CRC to a wildly different value — that's the guarantee we want.

---

## Trade-offs and Alternatives

| Decision                         | Option A                     | Option B                          | When A wins                                   | When B wins                                    |
|----------------------------------|------------------------------|-----------------------------------|-----------------------------------------------|------------------------------------------------|
| Record layout                    | Fixed-size slots             | Slotted pages (variable)          | All records same size; simple code            | Variable-size keys/values; deletions common    |
| Variable-field location          | Implicit (sequential scan)   | Explicit offset/length in header  | Few variable fields; sequential access        | Random access by field; many variable fields   |
| Free-space reuse                 | First-fit                    | Best-fit                          | Speed > packing; many similar-size inserts    | Tight packing matters; heterogeneous sizes     |
| Space reclamation                | Availability list (lazy)     | Eager defragmentation             | Frequent writes; CPU-bound                    | Space-bound; infrequent compactions OK         |
| Version identification           | Filename prefix              | In-header magic number            | Bulk file management; readable without open   | Files move around; single self-contained file  |
| Integrity check                  | Plain checksum (XOR/parity)  | CRC32 / CRC64                     | Never — CRCs are cheap enough                 | Always for on-disk pages                       |
| Key/value co-location on leaf    | Inline key+value cell        | Separate key area + value area    | Record fits comfortably; small working set    | Very small keys + scan-heavy workloads (locality) |

---

## Key Takeaways

1. **On-disk layout is the data structure.** Without `malloc`/`free` you are the allocator — page size, headers, and offsets are first-class design choices.
2. **Fix the primitives first.** Endianness, Pascal-style length prefixes, and bit-packed flags are the byte-level grammar every higher-level structure is built from.
3. **Slotted pages decouple logical order from physical order.** Cells go where they land; the slot array keeps the sort order. Deletion is a pointer edit, not a data move.
4. **Two cell shapes, one template.** Fixed header + variable key + variable value, with page-level metadata describing shape. Internal nodes point to pages by ID; leaves hold records.
5. **Deletion is lazy by default.** Availability lists + first-fit/best-fit buy you amortized space reuse; defragmentation is the fallback, overflow pages are the last resort.
6. **Version identification is part of the format.** Pick filename prefix, sidecar file, or in-header magic number — but pick *before* you ship v1, because migrating later is painful.
7. **Per-page CRCs are non-negotiable.** Cheap, scoped, and they catch the burst errors real hardware actually produces. Cryptographic hashes are a different tool for a different threat.

---

## See Also

- [[01-binary-encoding-primitives]]
- [[02-file-organization-principles]]
- [[03-slotted-pages]]
- [[04-cell-layout]]
- [[05-managing-free-space]]
- [[06-file-format-versioning]]
- [[07-checksums-and-crcs]]